# Part 2 — Building and Loading Graphs

*Written for idtrack 0.1.2.*

This part shows how to **build and keep** a graph for each supported organism:
- `homo_sapiens` (human)
- `mus_musculus` (mouse)
- `sus_scrofa` (pig)
- `macaca_mulatta` (rhesus macaque) and `macaca_fascicularis` (cynomolgus macaque)

Building a graph is the one expensive step. You do it **once per organism, snapshot release and externals configuration**; afterwards the graph file in your local repository loads in seconds, without the network. Every organism ships its externals configuration, so nothing has to be configured first (your own selection of databases is Part 6).

**Learning objectives**
- Build (or load) a graph for each organism.
- See what a graph holds and what its file records.
- Learn when a graph is loaded, built or built again.

## 2.0 — What to expect

A first build downloads Ensembl's tables of every release of the organism and builds the graph from them: from about 12 to 14 minutes for cynomolgus over HTTPS to about 2.3 to 2.6 hours plus the download for human. A graph file takes from 22.5 MB (cynomolgus) to 155.6 MB (human), and loads in 0.5 to 4 seconds. The measured times, sizes and memory per organism: [Time and memory](../performance.rst).

Plan a first build as you would plan downloading a reference genome and its annotation.

In [1]:
# Load notebook utilities (collapsible output magic for tutorials). `_notebook_utils.py` sits next to the notebooks in
# idtrack's repository; a notebook downloaded on its own runs without it and shows the logs as they come.
try:
    get_ipython().run_line_magic('load_ext', '_notebook_utils')
except ModuleNotFoundError:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def collapse(line, cell):
        get_ipython().run_cell(cell)

In [2]:
# 1) Setup
from __future__ import annotations

import os
from pathlib import Path

import idtrack
from idtrack import API

# The local repository (IDTRACK_LOCAL_REPO is a convention of these tutorials; idtrack reads no such variable).
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

API.configure_logger()

print('Local repository:', LOCAL_REPOSITORY)

Local repository: idtrack_cache


## 2.1 — Which externals configuration will be used?

Every supported organism ships its externals configuration (`idtrack/default_config/<organism>_externals_modified.yml`). A file of the same name in your local repository (Part 6) takes its place.

In [3]:
DEFAULT_CONFIG = Path(idtrack.__file__).parent / 'default_config'
ORGANISMS = ['homo_sapiens', 'mus_musculus', 'sus_scrofa', 'macaca_mulatta', 'macaca_fascicularis']

for organism in ORGANISMS:
    name = f'{organism}_externals_modified.yml'
    used = 'local repository' if (LOCAL_REPOSITORY / name).exists() else 'shipped with idtrack'
    print(f'{organism:<20} {name:<45} {used}')

homo_sapiens         homo_sapiens_externals_modified.yml           shipped with idtrack
mus_musculus         mus_musculus_externals_modified.yml           shipped with idtrack
sus_scrofa           sus_scrofa_externals_modified.yml             shipped with idtrack
macaca_mulatta       macaca_mulatta_externals_modified.yml         shipped with idtrack
macaca_fascicularis  macaca_fascicularis_externals_modified.yml    shipped with idtrack


## 2.2 — Build or load a graph

The pattern is:

1. resolve the organism's name;
2. fix the snapshot release (the last release of the graph);
3. `API.build(organism, snapshot_release, local_repository)`: it loads the graph file when the local repository holds it, and builds and saves it otherwise (`API.load(path)` loads a file and never builds);
4. inspect it, then convert (Part 3).

The tutorials fix the snapshot release at 115, so that their answers stay the same when Ensembl publishes a new release. `API.ensembl_organism(name)` asks Ensembl REST for the organism's formal name and Ensembl's newest release; `idtrack.organisms.resolve_organism(name)` gives the formal name offline, for the names of idtrack's own table.

The primary assembly of a graph is the organism's preferred assembly that has the snapshot release (human GRCh38, mouse GRCm39, pig Sscrofa11.1, rhesus Mmul_10, cynomolgus 6.0 at release 115); it is part of the graph file's name, which `API.graph_path(...)` gives.

The same from the command line: `idtrack build human 115 --repository idtrack_cache` ([Command line](../cli.rst)).

### 2.2.1 — Human (two assemblies in parallel)

The human graph has GRCh38 as its primary assembly (assembly key `38`). Conversions use, at each release, the assembly Ensembl maintained at that release (NCBI36 `36` to release 54, GRCh37 `37` to 75, GRCh38 from 76), and the GRCh37 ids of later releases are in the graph as ids to convert from. So datasets annotated on different genome builds come to one set of identifiers.

In [4]:
organism, latest_release = API.ensembl_organism('human')
SNAPSHOT_RELEASE = 115  # fixed: the same graph and the same answers in every run (latest_release is Ensembl's newest)
print(organism, 'snapshot release', SNAPSHOT_RELEASE, '| Ensembl newest release today:', latest_release)
print('Graph file:', API.graph_path(organism, SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY)))

2026-11-24 11:51:46 INFO:idtrack.build.species: Ensembl Rest API query to get the organism names and associated releases.


homo_sapiens snapshot release 115 | Ensembl newest release today: 116
Graph file: idtrack_cache/graph_homo_sapiens_assembly38_min48_max115.npz


In [5]:
%%collapse Click to show the log
# Build (or load) the graph
api = API.build(organism, SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))

In [6]:
# What the graph holds (read from the graph, no network)
described = api.describe()
print('Organism:', described.organism)
print('Snapshot release:', described.snapshot_release)
print('Primary assembly:', described.primary_assembly)
print('Assemblies in this graph:', sorted(a.assembly for a in described.assemblies))
print('Nodes:', described.counts['nodes'])
print('Edges:', described.counts['history_edges'] + described.counts['cross_reference_edges'])

names = api.external_databases()
print('External database names in the graph:', len(names))
print('First 12:', names[:12])

# What the graph file records about itself (checked when it is loaded)
record = api.graph.record
print('Graph format version:', described.format_version, '| built by idtrack', described.package_version)
print('Corrections of Ensembl history made while building:', record.history['corrections'])

2026-11-24 11:51:52 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


Organism: homo_sapiens
Snapshot release: 115
Primary assembly: 38
Assemblies in this graph: [36, 37, 38]
Nodes: 3715701
Edges: 9291635
External database names in the graph: 35
First 12: ['CCDS', 'CCDS label', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC (automatic)', 'HGNC (curated)', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'Havana translation', 'IMGT/GENE-DB']
Graph format version: 6 | built by idtrack none: built from a source folder, not an installed idtrack
Corrections of Ensembl history made while building: [{'edges': 7, 'events': 7, 'recorded': [114, 114], 'used': [114, 115]}]


How to read this:

- `external_databases()` lists the names of Ensembl's databases under which the graph holds external ids. These are more than the selected databases of the externals configuration: a selected database is, at every release, every database of Ensembl whose links continue it (HGNC's symbols, for example, are filed under several names across releases). [External databases](../databases.rst) explains the filing and which name to pass as `final_database`.
- "built by idtrack ..." names the version of idtrack that built the graph. The graphs of these outputs were built from a source checkout of this version, which records itself as "none: built from a source folder"; a graph you build with an installed release names its version there.
- The corrections are Ensembl's mapping sessions whose recorded releases disagree with its id tables; each is placed at the releases the id tables show (here a session recorded as releases 114 to 114, with 7 events, belongs to the step from 114 to 115).

### 2.2.2 — Mouse (one assembly per release)

Mouse has one maintained assembly per release (NCBIM37 → GRCm38 → GRCm39). Its older assemblies matter for older datasets; there are no two assemblies in one release.

In [7]:
%%collapse Click to show the log
api = API.build('mus_musculus', SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))

In [8]:
described = api.describe()
print('Primary assembly:', described.primary_assembly)
print('Assemblies in this graph:', sorted(a.assembly for a in described.assemblies))
print('Nodes:', described.counts['nodes'])
print('Edges:', described.counts['history_edges'] + described.counts['cross_reference_edges'])
print('External database names in the graph:', len(api.external_databases()))

2026-11-24 11:51:56 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


Primary assembly: 39
Assemblies in this graph: [37, 38, 39]
Nodes: 2470807
Edges: 5869690
External database names in the graph: 37


### 2.2.3 — Pig (one assembly per release)

Pig has one maintained assembly per release (Sscrofa9 → Sscrofa10.2 → Sscrofa11.1); its graph begins at release 56, pig's first release in idtrack.

In [9]:
%%collapse Click to show the log
api = API.build('sus_scrofa', SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))

In [10]:
described = api.describe()
print('Primary assembly:', described.primary_assembly)
print('Assemblies in this graph:', sorted(a.assembly for a in described.assemblies))
print('Nodes:', described.counts['nodes'])
print('Edges:', described.counts['history_edges'] + described.counts['cross_reference_edges'])
print('External database names in the graph:', len(api.external_databases()))

2026-11-24 11:51:57 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


Primary assembly: 111
Assemblies in this graph: [9, 102, 111]
Nodes: 1096854
Edges: 1804825
External database names in the graph: 24


### 2.2.4 — Rhesus and cynomolgus macaques (one assembly per release)

- rhesus (`macaca_mulatta`): MMUL_1 (key `1`, releases 48–85) → Mmul_8.0.1 (`801`, 86–97) → Mmul_10 (`10`, 98 on)
- cynomolgus (`macaca_fascicularis`): Macaca_fascicularis_5.0 (`5`, 91–102) → 6.0 (`6`, 103 on)

The graph links the assemblies through Ensembl's own history (rhesus 85 → 86 and 97 → 98, for example), so identifiers of an older genome build convert to the current one. Gene symbols are VGNC symbols on the current assemblies and HGNC symbols on the older ones; both are selected in the shipped configurations. Rhesus and cynomolgus are two organisms and two graphs.

In [11]:
%%collapse Click to show the log
api = API.build('macaca_mulatta', SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))
print('Assemblies in this graph:', sorted(a.assembly for a in api.describe().assemblies))

In [12]:
%%collapse Click to show the log
api = API.build('macaca_fascicularis', SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))
print('Assemblies in this graph:', sorted(a.assembly for a in api.describe().assemblies))

### 2.2.5 — Loading a graph file directly

`API.load(path)` reads a graph file and nothing else: no tables, no network, and not the build code. It is the call to use in an analysis once the graph exists (the command line's `idtrack convert <graph file> ...` does the same).

In [13]:
path = API.graph_path('macaca_fascicularis', SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))
api = API.load(path)
print(api.describe().organism, api.releases()[0], 'to', api.releases()[-1])

2026-11-24 11:51:59 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


macaca_fascicularis 91 to 115


## 2.3 — Managing graphs

### Loading, building and building again

- `API.build(...)` is safe to call repeatedly. If the graph file is in the local repository, it is **loaded** from the file alone (no tables, no network). If not, the graph is **built** (slow, the first time only) and saved.
- `API.build(..., rebuild=True)` (command line: `idtrack build ... --rebuild`) builds the graph even if its file exists, and replaces the file. Tables already in the local repository are not downloaded again.
- One `idtrack.API` holds one graph (one organism, one snapshot release). Keep one `API` per organism if you need several at once.

### What a graph file is

- One `.npz` file per graph, `graph_<organism>_assembly<primary assembly>_min<first release>_max<snapshot release>.npz`: graphs of different organisms, snapshot releases and primary assemblies sit side by side in one local repository.
- The file records what it was built from: the selection of external databases (from the externals configuration), the organism table, the graph format and the package version, the tables, and the corrections made to Ensembl's history (`api.describe()`; the whole record in `api.graph.record`). Loading a file under the expected name that was built from something else (another selection, another organism table, an older graph format) raises `idtrack.GraphMismatchError`, which names the differences: idtrack never answers from another graph and never starts a build of hours on its own. Build it again with `rebuild=True`.
- Graph files of idtrack 0.0.5 (`graph_*.pickle`) and its table cache (`<organism>_assembly-<key>.h5`) are not read; build the graphs again with this version.

### What the local repository holds

- downloaded tables (`idtrack_tables/<organism>_assembly-<key>/`, one file per table; a table records what it depends on, so changing the externals configuration or the window never serves a stale table and nothing has to be deleted by hand);
- graph files (`graph_*.npz`);
- your own externals configurations, if you made any (Part 6).

> **Tip:** Treat your local repository as project infrastructure: keep it in a stable place, so that tables and graphs are reused. Deleting its tables means downloading them again for the next build.

### A graph is specific to

- the organism;
- the window: its first release and its snapshot release;
- the selection of external databases (`<organism>_externals_modified.yml` of the local repository, or the shipped one).

To compare two selections of external databases, use one local repository per selection (the configuration's file name is fixed per repository; Part 6).

In [14]:
# List what is in the local repository (deletes nothing).
print('Local repository:', LOCAL_REPOSITORY)
for pattern in ['graph_*.npz', 'idtrack_tables/*', '*_externals_modified.yml']:
    hits = sorted(LOCAL_REPOSITORY.glob(pattern))
    print(f'\n{pattern} ({len(hits)}):')
    for p in hits[:10]:
        print('  ', p.name)

Local repository: idtrack_cache

graph_*.npz (5):
   graph_homo_sapiens_assembly38_min48_max115.npz
   graph_macaca_fascicularis_assembly6_min91_max115.npz
   graph_macaca_mulatta_assembly10_min48_max115.npz
   graph_mus_musculus_assembly39_min48_max115.npz
   graph_sus_scrofa_assembly111_min56_max115.npz

idtrack_tables/* (0):

*_externals_modified.yml (0):
